# 01 · ERA5 raw → daily SWH

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import swh_core as core

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES    = ROOT / "output" / "tables"
FIGURES   = ROOT / "output" / "figures"
for d in (PROCESSED, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": False})
pd.set_option("display.width", 120)

WPP = "573"

TAHUN_AWAL, TAHUN_AKHIR = core.tahun_lengkap_terakhir()

print(ROOT)
print(f"jendela kalibrasi: {TAHUN_AWAL}-{TAHUN_AKHIR}")

## files

In [ ]:
semua = sorted(RAW.glob(f"*wpp{WPP}*.nc"))
if not semua:
    raise SystemExit(f"Belum ada data di {RAW}. "
                     f"Jalankan: python src/download_era5_swh.py --wpp {WPP}")

files, luar = [], []
for f in semua:
    th = core.tahun_dari_nama(f)
    dalam = (th is None or TAHUN_AWAL is None or TAHUN_AWAL <= th <= TAHUN_AKHIR)
    (files if dalam else luar).append(f)

if not files:
    print(f"! Tidak ada file dalam jendela {TAHUN_AWAL}-{TAHUN_AKHIR}.")
    print("  Sementara pakai semua file yang ada (mode uji coba).\n")
    files, luar = semua, []

for f in files:
    print(f"{f.name:45s} {f.stat().st_size/1024:8.0f} KB")
print(f"\n{len(files)} file dipakai" + (f", {len(luar)} di luar jendela" if luar else ""))

## one file

In [ ]:
ds = core.open_any(files[0])
ds

In [ ]:
da = core.pick_var(ds)

print("dimensi   :", dict(da.sizes))
print("lintang   :", float(da.latitude.min()), "s/d", float(da.latitude.max()))
print("bujur     :", float(da.longitude.min()), "s/d", float(da.longitude.max()))
print("resolusi  :", float(abs(da.latitude.diff('latitude')[0])), "derajat")
print("satuan    :", da.attrs.get("units", "?"))

total = da.isel(valid_time=0).size
laut  = int(da.isel(valid_time=0).notnull().sum())
print(f"\nsel grid  : {total} total, {laut} laut, {total-laut} darat (NaN, otomatis diabaikan)")

### mean map (bbox check)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
m = da.mean("valid_time")
im = ax.pcolormesh(m.longitude, m.latitude, m.values, shading="auto", cmap="YlGnBu")
fig.colorbar(im, ax=ax, label="SWH rata-rata (m)")
ax.set_xlabel("Bujur"); ax.set_ylabel("Lintang")
ax.set_title(f"Rata-rata SWH maksimum harian — WPP {WPP}")
plt.show()

## grid → one number per day

In [ ]:
df = core.build_series(files)
print(f"{len(df):,} hari — {df.tanggal.min().date()} s/d {df.tanggal.max().date()}")
df.head(10)

In [ ]:
df[["swh_mean", "swh_p90", "swh_max"]].describe().round(3)

## series + tier thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(df.tanggal, df.swh_mean, lw=0.8, color="#1f4e79", label="indeks (rata-rata WPP)")
ax.fill_between(df.tanggal, df.swh_mean, df.swh_max, color="#1f4e79", alpha=0.12, lw=0,
                label="s/d maks antar-grid")
for nama, a in core.AMBANG_TIER.items():
    ax.axhline(a, ls="--", lw=0.8, color="#b00020", alpha=0.7)
    ax.annotate(f"{nama} ({a} m)", xy=(1.002, a), xycoords=("axes fraction", "data"),
                fontsize=7, va="center", color="#b00020")
ax.set_ylabel("SWH maksimum harian (m)")
ax.set_title(f"Gambar 3.1 — SWH harian WPP {WPP} ({core.WPP_NAMA[WPP]})", fontsize=10)
ax.legend(fontsize=7, frameon=False, loc="upper left")
ax.margins(x=0.01)
plt.show()

## monthly counts

In [ ]:
AMBANG = 2.0

cacah = core.cacah_bulanan(df, AMBANG)
cacah.head(12)

### seasonality

In [ ]:
if cacah.bulan.nunique() >= 6:
    rata = cacah.groupby("bulan").n_terpicu.mean()
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(rata.index, rata.values, color="#1f4e79", width=0.7)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"])
    ax.set_ylabel(f"Rata-rata hari SWH > {AMBANG} m")
    ax.set_title(f"Gambar 3.2 — Pola musiman frekuensi pemicu, WPP {WPP}", fontsize=10)
    plt.show()
else:
    print(f"Baru {cacah.bulan.nunique()} bulan tersedia — pola musiman belum bisa dinilai.")
    print("Jalankan unduhan penuh dulu: python src/download_era5_swh.py --wpp", WPP)

## threshold vs payout frequency

In [ ]:
tabel = core.ringkas_ambang(df)
tabel.round(2)

## save

In [ ]:
SUFFIX = "_NB"

out1 = PROCESSED / f"swh_harian_wpp{WPP}{SUFFIX}.csv"
out2 = PROCESSED / f"cacah_bulanan_wpp{WPP}{SUFFIX}.csv"
out3 = TABLES    / f"ambang_vs_frekuensi_wpp{WPP}{SUFFIX}.csv"

df.to_csv(out1, index=False, float_format="%.4f")
cacah.to_csv(out2, index=False, float_format="%.4f")
tabel.to_csv(out3, index=False, float_format="%.3f")

for p in (out1, out2, out3):
    print("tersimpan:", p.relative_to(ROOT))